# LSTM Cell and Forward Pass
Notebook: `d21_s3_a1.ipynb`


In [1]:
import numpy as np
rng = np.random.RandomState(0)
D, H = 3, 4
Wx = rng.randn(4*H, D) * 0.5
Wh = rng.randn(4*H, H) * 0.5
b = rng.randn(4*H) * 0.3
X = rng.randn(6, D)


In [2]:
def sigmoid(x):
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x)
    pos = x >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-x[pos]))
    exp_x = np.exp(x[~pos])
    out[~pos] = exp_x / (1.0 + exp_x)
    return out

def lstm_cell_step(x_t, h_prev, c_prev, Wx, Wh, b):
    H = h_prev.shape[0]
    z = Wx @ x_t + Wh @ h_prev + b
    z_i = z[0:H]
    z_f = z[H:2*H]
    z_g = z[2*H:3*H]
    z_o = z[3*H:4*H]
    i = sigmoid(z_i)
    f = sigmoid(z_f)
    g = np.tanh(z_g)
    o = sigmoid(z_o)
    c_t = f * c_prev + i * g
    h_t = o * np.tanh(c_t)
    return h_t, c_t

def lstm_forward(X, h0, c0, Wx, Wh, b):
    h = h0.copy()
    c = c0.copy()
    hidden_states = []
    for x_t in X:
        h, c = lstm_cell_step(x_t, h, c, Wx, Wh, b)
        hidden_states.append(h)
    return np.array(hidden_states), c


In [3]:
print(np.round(sigmoid(np.array([0.0, 2.0])), 4))
X = rng.randn(6, 3); h0 = np.zeros(4); c0 = np.zeros(4)
H_all, c_last = lstm_forward(X, h0, c0, Wx, Wh, b)
print(H_all.shape, c_last.shape)
b2 = np.zeros(16); b2[0:4] = -30; b2[4:8] = +30
_, c_t = lstm_cell_step(np.zeros(3), np.zeros(4), np.array([1.,2.,3.,4.]), np.zeros((16,3)), np.zeros((16,4)), b2)
print(np.round(c_t, 4))


[0.5    0.8808]
(6, 4) (4,)
[1. 2. 3. 4.]


In [4]:
# Autograder-style sanity checks
assert np.allclose(sigmoid(np.array([-1000., 0., 1000.])), [0., 0.5, 1.])
ht, ct = lstm_cell_step(X[0], h0, c0, Wx, Wh, b)
assert ht.shape == (4,) and ct.shape == (4,)
# Input overwrite: forget~0, input~1, candidate~1
b3 = np.zeros(16); b3[0:4] = 30; b3[4:8] = -30; b3[8:12] = 30; b3[12:16] = 30
_, c_over = lstm_cell_step(np.zeros(3), np.zeros(4), np.array([7.,8.,9.,10.]), np.zeros((16,3)), np.zeros((16,4)), b3)
assert np.allclose(c_over, np.ones(4), atol=1e-10)
print('All additional checks passed.')


All additional checks passed.
